In [2]:
!pip install -Uqq fastbook

In [3]:
import fastbook
fastbook.setup_book()

In [4]:
from fastbook import *

In [5]:
from fastai.text.all import *

In [6]:
path = untar_data(URLs.AMAZON_REVIEWS_POLARITY)

In [7]:
path.ls()

(#3) [Path('/root/.fastai/data/amazon_review_polarity_csv/test.csv'),Path('/root/.fastai/data/amazon_review_polarity_csv/readme.txt'),Path('/root/.fastai/data/amazon_review_polarity_csv/train.csv')]

In [8]:
print((path / "readme.txt").read_text())

Amazon Review Polaridy Dataset

Version 3, Updated 09/09/2015

ORIGIN

The Amazon reviews dataset consists of reviews from amazon. The data span a period of 18 years, including ~35 million reviews up to March 2013. Reviews include product and user information, ratings, and a plaintext review. For more information, please refer to the following paper: J. McAuley and J. Leskovec. Hidden factors and hidden topics: understanding rating dimensions with review text. RecSys, 2013.

The Amazon reviews polarity dataset is constructed by Xiang Zhang (xiang.zhang@nyu.edu) from the above dataset. It is used as a text classification benchmark in the following paper: Xiang Zhang, Junbo Zhao, Yann LeCun. Character-level Convolutional Networks for Text Classification. Advances in Neural Information Processing Systems 28 (NIPS 2015).


DESCRIPTION

The Amazon reviews polarity dataset is constructed by taking review score 1 and 2 as negative, and 4 and 5 as positive. Samples of score 3 is ignored. In th

In [11]:
train_pd = pd.read_csv(path/"train.csv")

In [7]:
train_pd.head()

,2,Stuning even for the non-gamer,This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^
0,2,The best soundtrack ever to anything.,"I'm reading a lot of reviews saying that this is the best 'game soundtrack' and I figured that I'd write a review to disagree a bit. This in my opinino is Yasunori Mitsuda's ultimate masterpiece. The music is timeless and I'm been listening to it for years now and its beauty simply refuses to fade.The price tag on this is pretty staggering I must say, but if you are going to buy any cd for this much money, this is the only one that I feel would be worth every penny."
1,2,Amazing!,"This soundtrack is my favorite music of all time, hands down. The intense sadness of ""Prisoners of Fate"" (which means all the more if you've played the game) and the hope in ""A Distant Promise"" and ""Girl who Stole the Star"" have been an important inspiration to me personally throughout my teen years. The higher energy tracks like ""Chrono Cross ~ Time's Scar~"", ""Time of the Dreamwatch"", and ""Chronomantique"" (indefinably remeniscent of Chrono Trigger) are all absolutely superb as well.This soundtrack is amazing music, probably the best of this composer's work (I haven't heard the Xenogears s..."
2,2,Excellent Soundtrack,"I truly like this soundtrack and I enjoy video game music. I have played this game and most of the music on here I enjoy and it's truly relaxing and peaceful.On disk one. my favorites are Scars Of Time, Between Life and Death, Forest Of Illusion, Fortress of Ancient Dragons, Lost Fragment, and Drowned Valley.Disk Two: The Draggons, Galdorb - Home, Chronomantique, Prisoners of Fate, Gale, and my girlfriend likes ZelbessDisk Three: The best of the three. Garden Of God, Chronopolis, Fates, Jellyfish sea, Burning Orphange, Dragon's Prayer, Tower Of Stars, Dragon God, and Radical Dreamers - Uns..."
3,2,"Remember, Pull Your Jaw Off The Floor After Hearing it","If you've played the game, you know how divine the music is! Every single song tells a story of the game, it's that good! The greatest songs are without a doubt, Chrono Cross: Time's Scar, Magical Dreamers: The Wind, The Stars, and the Sea and Radical Dreamers: Unstolen Jewel. (Translation varies) This music is perfect if you ask me, the best it can be. Yasunori Mitsuda just poured his heart on and wrote it down on paper."
4,2,an absolute masterpiece,"I am quite sure any of you actually taking the time to read this have played the game at least once, and heard at least a few of the tracks here. And whether you were aware of it or not, Mitsuda's music contributed greatly to the mood of every single minute of the whole game.Composed of 3 CDs and quite a few songs (I haven't an exact count), all of which are heart-rendering and impressively remarkable, this soundtrack is one I assure you you will not forget. It has everything for every listener -- from fast-paced and energetic (Dancing the Tokage or Termina Home), to slower and more haunti..."


In [18]:
def get_amazon_sample(path):
    path = Path(path)
    samples = []

    for filename in ("train.csv", "test.csv"):
        df = pd.read_csv(
            path / filename,
            header=None,
            names=["label", "title", "text"],
            keep_default_na=False,
        )

        # Remember which rows belong in the validation set.
        df["is_valid"] = filename == "test.csv"

        # Keep the same repeatable 10% sample of reviews.
        sample_size = max(1, int(len(df) * 0.0005))
        samples.append(
            df.sample(n=sample_size, random_state=42)
        )

    return pd.concat(samples, ignore_index=True)

In [19]:
dls_lm = DataBlock(
    blocks=TextBlock.from_df(
        "text",
        is_lm=True,
        n_workers=0,
    ),
    get_items=get_amazon_sample,
    get_x=ColReader("text"),
    splitter=RandomSplitter(0.1, seed=42),
).dataloaders(
    path,
    path=path,
    bs=32,
    val_bs=32,
    seq_len=80,
    num_workers=0,
)

In [10]:
dls_lm.show_batch(max_n=5)

,text,text_
0,"xxbos i wear a size 1x sometimes a 2x , so i ordered this in the xxunk size . xxmaj the waist was so small that i would have had to buy a xxunk to get it to fit . xxmaj this only good for someone who does n't really need a waist shaper . xxbos xxmaj nice one but … .after 3 month light use , this camera has a crack on the xxup lcd . i do n't","i wear a size 1x sometimes a 2x , so i ordered this in the xxunk size . xxmaj the waist was so small that i would have had to buy a xxunk to get it to fit . xxmaj this only good for someone who does n't really need a waist shaper . xxbos xxmaj nice one but … .after 3 month light use , this camera has a crack on the xxup lcd . i do n't remember"
1,xxmaj it was one of my favorites when i was a child and i still as an adult enjoy this movie xxrep 5 ! xxbos xxmaj this stuff is great ! i love to buy fresh salsa but it does n't last long . xxmaj this stuff comes out of a jar but it tastes like the fresh stuff from the refrigerated section . xxbos xxmaj just when you think that your head is going to explode if you see,it was one of my favorites when i was a child and i still as an adult enjoy this movie xxrep 5 ! xxbos xxmaj this stuff is great ! i love to buy fresh salsa but it does n't last long . xxmaj this stuff comes out of a jar but it tastes like the fresh stuff from the refrigerated section . xxbos xxmaj just when you think that your head is going to explode if you see one
2,"probably would n't be so big , but you have to download the whole of xxmaj north xxmaj america , to include xxmaj canada and xxunk , for five - hundred dollars , i got seven - year - old maps that i ca n't upgrade . i probably could have bought a seven - year - old atlas for xxunk cents . i will be running back to xxmaj garmin now . xxbos "" reservoir xxmaj stimulation "" is","would n't be so big , but you have to download the whole of xxmaj north xxmaj america , to include xxmaj canada and xxunk , for five - hundred dollars , i got seven - year - old maps that i ca n't upgrade . i probably could have bought a seven - year - old atlas for xxunk cents . i will be running back to xxmaj garmin now . xxbos "" reservoir xxmaj stimulation "" is not"
3,"few pages ( only xxunk ) . xxmaj however , i think the book also have very poor linkage between the book content and example source code . xxmaj the book explains and traces the source code well . xxmaj on the other hand , most likely , you wo n't know which source file to look into , how to compile , or how to run ( of course , in few cases , the authors will tell you","pages ( only xxunk ) . xxmaj however , i think the book also have very poor linkage between the book content and example source code . xxmaj the book explains and traces the source code well . xxmaj on the other hand , most likely , you wo n't know which source file to look into , how to compile , or how to run ( of course , in few cases , the authors will tell you )"
4,". xxmaj boy , was i wrong . xxmaj when i tried to put this under my infant carseat it brought the carseat so far away from the back of the seat that i was unable to safely buckle it . xxmaj there was no way to get a tight fit for the carseat using the 1 inch rule . ( the carseat should not move more than 1 inch in any direction ) i again tried to use the","xxmaj boy , was i wrong . xxmaj when i tried to put this under my infant carseat it brought the carseat so far away from the back of the seat that i was unable to safely buckle it . xxmaj there was no way to get a tight fit for the carseat using the 1 inch rule . ( the carseat should not move more than 1 inch in any direction ) i again tried to use the leveler"


# Fine-Tuning the Language Model

In [20]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3, 
    metrics=[accuracy, Perplexity()]).to_fp16()

In [21]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,4.058396,3.968760,0.257750,52.918896,00:11


In [22]:
learn.save_encoder('finetuned')

# Creating the Classifier DataLoaders

In [23]:
dls_clas = DataBlock(
    blocks=(
        TextBlock.from_df(
            "text",
            vocab=dls_lm.vocab,
            seq_len=72,
            n_workers=0,
        ),
        CategoryBlock,
    ),
    get_items=get_amazon_sample,
    get_x=ColReader("text"),
    get_y=ColReader("label"),
    splitter=ColSplitter("is_valid"),
).dataloaders(
    path,
    path=path,
    bs=32,
    val_bs=32,
    num_workers=0,
)

In [24]:
dls_clas.show_batch(max_n=3)

,text,category
0,"xxbos xxup if you 've ever wondered what "" mr . & xxmaj mrs . xxmaj smith "" ( or , the xxup tv shows "" xxunk & xxmaj mrs . xxmaj king "" and / or "" hart xxmaj to xxmaj hart "" ) would be like xxunk of all wit & charm , "" killers "" is the result . xxmaj mr . xxup a. xxmaj xxunk is xxup still pretty much playing xxmaj xxunk ( "" that 70s xxmaj show "" ) and never for a second did i believe him as a pro killer . xxmaj ms . xxmaj xxunk h shows just how limited her acting range is . xxmaj the action scenes are nothing special and the dialogue is fairly lame . i xxup know it 's a ' popcorn movie ' and not meant to be taken ( too ) seriously , but",1
1,xxbos i loved xxup xxunk movie and would tell all my friends about it . xxmaj but this … .this is crap . xxmaj for some reason i just had to read the first 3 books but now i have xxunk xxunk reading . xxmaj the character development is very good but one thing that keeps this from being good . xxup its xxup way xxup to xxup graphic xxup for xxup its xxup taste ! i know it says xxmaj xxunk on the back but still jeez ! i mean you see some much xxunk and xxunk get shot or pulled out . xxmaj for an example xxmaj xxunk xxunk brother gets shot in the xxunk and you see a graphic picture of him xxunk on the ground with a huge whole with his xxunk xxunk out . xxmaj and than the teacher xxunk him off by shooting him,1
2,"xxbos boy , is this guy overrated . do n't believe the hype ( if there is any and then it will soon go away -- o xxrep 3 h xxunk 'em ! ) . i was told to let the record xxunk in and that i would not like it right away , well i 've been xxunk for a whole week … and it 's still xxunk . if you like xxunk milk hotel and xxunk and xxunk , which bright eyes gets compared to , than you will be dissapointed . it 's like if you like pearl jam and then said "" i like them s xxrep 4 o much that i think i 'll buy all the creed albums ! "" bright eyes is a copy of a copy of a copy . the lyrics are like bad poetry from xxunk 's ( of pearl",1


In [25]:
learn = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5, 
                                metrics=accuracy).to_fp16()

In [26]:
learn = learn.load_encoder('finetuned')

In [27]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,time
0,0.558018,0.435511,0.830000,00:06


In [28]:
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

epoch,train_loss,valid_loss,accuracy,time
0,0.512416,0.394488,0.845000,00:07


In [29]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.428961,0.316670,0.885000,00:10


In [30]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.337386,0.318051,0.865000,00:12
1,0.290854,0.312330,0.885000,00:12


# Make predictions, negative then positive

In [31]:
learn.predict(
    "These headphones stopped working after two days. "
    "The sound was terrible even before they broke. "
    "A complete waste of money."
)

('1', tensor(0), tensor([0.9948, 0.0052]))

In [32]:
learn.predict(
    "These headphones are comfortable and sound fantastic. "
    "The battery lasts all day, and they were easy to set up. "
    "I would happily buy them again."
)

('2', tensor(1), tensor([0.0015, 0.9985]))